## DSPy Ollama Qwen2 Information Extraction Pydantic

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rich import print
import pandas as pd
import ast

from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch
from collections.abc import Iterable


from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer
from pydantic import BaseModel

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2', 'rougeL'], use_stemmer=True)
import json

from data.train_examples import train_example_list
from data.valid_examples import dev_example_list
from data.test_example import test_examples_list

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


#### Helper Functions

In [19]:

def validate_ans(example, pred, trace = None):
    
    job_dict = {"position_title": example.position_title,
                    "location" : example.location,
                    "work_arrangement": example.work_arrangement,
                    "experience": example.experience,
                    "employment_type": example.employment_type,
                    "pay": example.pay,
                    "degree": example.degree,
                    "certifications": example.certifications,
                    "required_skills": example.required_skills,}
     

    gold = re.sub(r'\n|\s+ ', '',str(job_dict)).lower()
    print(gold)

    prediction = str(pred.info_extracted).lower()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Data

In [3]:
train_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet1.csv')

dev_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/50examples_for_DSPy_withJson.csv')

test_examples = pd.read_csv("/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet2.csv", header = None)

#### Set Ollama LLM using Qwen 2 from Alibaba

In [4]:
llm = dspy.OllamaLocal(model='qwen2:latest', max_tokens = 1000, temperature=0.0)
dspy.settings.configure(lm=llm)

#### Set Pydantic Class

In [5]:
class JobPostingExtractionCert(BaseModel):
    certifications : list[str] 
    
class JobPostingExtractionPay(BaseModel):
    pay : str

class JobPostingExtractionPostion(BaseModel):
    position_title: str

class JobPostingExtractionLocation(BaseModel):
    location : str

class JobPostingExtractionWorkArrange(BaseModel):
    work_arrangement: str 
    
class JobPostingExtractionExp(BaseModel):
    experience : str

class JobPostingExtractionEmpType(BaseModel):
    employment_type: str
    
class JobPostingExtractionDegree(BaseModel):
    degree : str

class JobPostingExtractionSkills(BaseModel):
    required_skills : list[str]

#### Create DSPy Signature 

In [6]:
class InfoExtractorCert(dspy.Signature):
    """Extracts single certifications and credentials from a job posting not education if nothing found put not specified. short factoids. cannot be more than 20 words or 20 characters
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_cert: JobPostingExtractionCert = dspy.OutputField(desc = "a list of strings related to certifications total should be less than 20 words or 20 characters")

class InfoExtractorPay(dspy.Signature):
    """Extracts salary or pay information from job posting if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_pay: JobPostingExtractionPay = dspy.OutputField(desc = "a string of salary or pay information per year or per hour has to be less than 5 words")

class InfoExtractorPostion(dspy.Signature):
    """Extracts the title or name of the position from a job posting if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_pos: JobPostingExtractionPostion = dspy.OutputField(desc = "3 to 5 words about the title of the job position has to be less than 5 words")


class InfoExtractorLocation(dspy.Signature):
    """Extracts where the job is located in the United States if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_loc: JobPostingExtractionLocation = dspy.OutputField(desc = "Where the job is located City State and ZipCode has to be less than 5 words")


class InfoExtractorWork(dspy.Signature):
    """Extracts information if the job is remote, hybrid, on-site if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_work: JobPostingExtractionWorkArrange = dspy.OutputField(desc = "4 to 5 words on if the job is remote hybrid or onsite has to be less than 3 words")

class InfoExtractorExp(dspy.Signature):
    """Extracts information on relevant years of experience required for a job if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_exp: JobPostingExtractionExp = dspy.OutputField(desc = "5 to 8 words on years of experience needed has to be less than 5 words")

class InfoExtractorEmpType(dspy.Signature):
    """Extracts information if the job is full-time part-time contract internship if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_emp: JobPostingExtractionEmpType = dspy.OutputField(desc = "5 to 8 words about the job type has to be less than 2 words")

class InfoExtractorDeg(dspy.Signature):
    """Extracts education or university information from job posting if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_deg: JobPostingExtractionDegree = dspy.OutputField(desc = "5 to 8 words wheter its bachelors, masters, high school or PHD has to be less than 5 words")

class InfoExtractorSkills(dspy.Signature):
    """Extracts relevant skills needed to perform job should not be more than 20 words or 20 characters if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_skills: JobPostingExtractionSkills = dspy.OutputField(desc = "total should not be more than 20 words or 20 characters")

#### Crease DSPy Module

In [7]:
class JobPostingModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.info_extraction_modules = {
            "position_title": dspy.ChainOfThoughtWithHint(InfoExtractorPostion, max_tokens=15,),
            "location": dspy.ChainOfThoughtWithHint(InfoExtractorLocation, max_tokens=15),
            "work_arrangement": dspy.ChainOfThoughtWithHint(InfoExtractorWork, max_tokens=30),
            "experience": dspy.ChainOfThoughtWithHint(InfoExtractorExp, max_tokens=15),
            "employment_type": dspy.ChainOfThoughtWithHint(InfoExtractorEmpType, max_tokens=15),
            "pay": dspy.ChainOfThoughtWithHint(InfoExtractorPay, max_tokens=15),
            "degree": dspy.ChainOfThoughtWithHint(InfoExtractorDeg, max_tokens=15),
            "certifications": dspy.ChainOfThoughtWithHint(InfoExtractorCert, max_tokens=30),
            "required_skills": dspy.ChainOfThoughtWithHint(InfoExtractorSkills, max_tokens=30)
        }

        self.attribute_names = {
            "position_title": "info_extracted_pos",
            "location": "info_extracted_loc",
            "work_arrangement": "info_extracted_work",
            "experience": "info_extracted_exp",
            "employment_type": "info_extracted_emp",
            "pay": "info_extracted_pay",
            "degree": "info_extracted_deg",
            "certifications": "info_extracted_cert",
            "required_skills": "info_extracted_skills"
        }

        self.hints = {
            "certifications": "Usually 3 or 4 letters capitalized, not an academic degree like BS, Masters or PHD but something you obtain professionally or through job experience or relevant technologies or certificates",
            "pay": "The dollars per hour or salary for the year or pay",
            "position_title": "The title of the job or position of the job",
            "location": "Where the job is located in the United States",
            "work_arrangement": "If the job is hybrid, remote on-site or where the job is located to go into the office",
            "experience": "Number of years of experience that a job posting is asking for",
            "employment_type": "full time, part time, contract or internship or apprenticeship",
            "degree": "Education level such as GED, High School, Bachelors, Masters, Doctorate, PHD",
            "required_skills": "The required skills to do the job"
        }

    def forward(self, job_posting):
        job_posting = job_posting.replace('\n', ' ').replace('“', '"').replace('”', '"')
        job_posting = normalize(job_posting)

        job_dict = {}
        for key in ["position_title", "location", "work_arrangement", "experience", "employment_type", "pay", "degree", "certifications", "required_skills"]:
            extraction_module = self.info_extraction_modules[key]
            hint_value = self.hints[key]
            extracted_info = getattr(extraction_module(job_posting=job_posting, hint=hint_value), self.attribute_names[key]).replace("```\n", "").replace("```", "")
            dspy.Suggest(len(extracted_info) <= 400,f"info extract should be short and less than 400 characters right now its {len(extracted_info)} for {key}",)
            job_dict[key] = extracted_info

        return dspy.Prediction(job_posting=job_posting, info_extracted=job_dict)


In [8]:
uncompiled_module = JobPostingModule()

#### Perform Test Extraction on Data with uncompiled version

In [9]:
print(test_examples[1][5])


EMT-Advanced-Emergency Medical Service
Job Locations
US-TX-Rosenberg
Posted Date
9 months ago
(5/16/2023 11:01 AM)
ID 2023-5265 Job Code DJOB # of Openings 10 Min Start Salary USD $2,008.28/Bi. Category EMS Max Start Salary USD 
$2,421.41/Bi.
Overview

Fort Bend County is ranked as one of the fastest growing counties in the nation. We have capitalized on not only 
the creed of our location, but on the "quality of life" for our families to call home. Our employees are the key to
our success and the heartbeat of our foundation. The diversity and inclusivity of our community is our strength and
at the forefront of a workplace environment welcoming to all. Live Here! Work Here!



Provides emergency medical care to the citizens of Fort Bend County as stated in established standards and 
procedures.



Responsibilities
Provides emergency pre-hospital medical care. 
Completes reports within established timeframes.
Maintains emergency vehicle(s) and inventory of medical supplies.
Signs for and is held accountable for equipment issued and used.
Operates emergency vehicles (i.e. ambulance, squad) per department policy and with due regard to the law.
Responsible for maintaining all current certifications within department guidelines as required.
Certifications shall include EMT-Basic that has successfully completed AEMT and is test eligible and is enrolled in
an EMT Paramedic Program, DSHS EMT-Advanced Certification who is enrolled in an EMT Paramedic Program, and Valid 
State of Texas Driver's License.
Prepares, submits, and maintains clear, concise, and accurate documentation on patient care activities, incident 
reports and other related information as requested.
Assists other employees with their duties.
Performs general housekeeping duties for station and department areas.
Participates in activities and duties related to emergency management during a local state of disaster as directed 
by appropriate county managers.
Qualifications
High School Diploma/GED; Enrolled in College pursing Paramedic Certification and/or EMS Degree.
Certified or Licensed State of Texas EMT-Basic or EMT-Advanced or is eligible to test for EMT-Advanced 
Certification. 
Current Healthcare Provider CPR/AED card.
Pre-hospital experience preferred.
Experience in a high performance ALS system beneficial.
Strong verbal and written communication and organizational skills.
Strong interpersonal skills and ability to deal effectively with the public and other employees. 
Frequent reading, writing, memorization, analyzing, simple math skills, negotiating. Constantly using judgment, 
reasoning, decision-making and teaching.
Ability to complete projects.
Must obtain and maintain a current American Heart Association Advanced Cardiac Life Support certification.
Must complete National Incident Management System (NIMS) 100, 200, 700 and 800 within 90 days of hire.
Must obtain Paramedic Credentials in 24 months after hire date. Subject to emergency call-in and mandatory 
staffing.



SALARY RANGE: EMS Grade EMT-1, $2,008.28 - $2,421.41 biweekly based on qualifications

CLOSING DATE: Upon filling position





Fort Bend County is an equal opportunity employer, committed to non-discrimination in employment on any basis 
including race, color, religion or creed, sex, sexual orientation, gender, gender identity, gender expression, 
pregnancy status (including childbirth and related medical conditions), national origin, ethnicity, citizenship 
status, age (40 and over), physical or mental disability, genetic information, protected military and veteran 
status, political affiliation or beliefs, or any other classification protected by state, federal and local laws, 
unless such classification is a bona fide occupational qualification.

In [10]:
with dspy.context(lm = llm):
    pred = uncompiled_module(job_posting =test_examples[1][5])
    print(pred.info_extracted)

{
    'position_title': 'Emergency Medical Technician',
    'location': 'Rosenberg, TX',
    'work_arrangement': 'On-site',
    'experience': 'Not specified',
    'employment_type': 'Full-time',
    'pay': '$2,008.28 - $2,421.41 biweekly',
    'degree': 'High School',
    'certifications': 'EMT-Advanced, DSHS EMT-Advanced, CPR/AED, ACLS, NIMS 100,200,700,800',
    'required_skills': "Emergency care, report writing, vehicle maintenance, inventory management, equipment 
accountability, CPR/AED, EMT certification, driver's license, documentation, communication, organization, public 
interaction, math proficiency, judgment, reasoning, teaching, project completion, ACLS, NIMS training, paramedic 
credentials, emergency readiness."
}

#### Inspect History and save DSPy program 

In [11]:
#print(llm.inspect_history(n=1))
uncompiled_module.save("uncompiled_file_pydantic_v2.json")

#### Create Training Examples, Validation(Dev) and Test Examples

In [12]:
print(len(dev_example_list) , len(train_example_list), len(test_examples_list))
train_results = train_example_list
train_contents = list(train_examples['body'])

dev_results = dev_example_list
dev_contents = list(dev_examples.loc[:20,'body'])

test_results = test_examples_list
test_contents = list(test_examples[1])

21 20 10

In [13]:
train_examples_list = [
    dspy.Example(
        job_posting=content,
        position_title=str(pos['position_title']),
        location=str(loc['location']),
        work_arrangement=str(work['work_arrangement']),
        experience=str(exp['experience']),
        employment_type=str(emp['employment_type']),
        pay=str(pay['pay']),
        degree=str(deg['degree']),
        certifications=str(cer['certification']),
        required_skills=str(req['required_skills'])
    )
    for content, pos, loc, work, exp, emp, pay, deg, cer, req in zip(
        train_contents, *([[json.loads(x) for x in train_results]] * 9)
    )
]

dev_examples_list = [
    dspy.Example(
        job_posting=content,
        position_title=str(pos['position_title']),
        location=str(loc['location']),
        work_arrangement=str(work['work_arrangement']),
        experience=str(exp['experience']),
        employment_type=str(emp['employment_type']),
        pay=str(pay['pay']),
        degree=str(deg['degree']),
        certifications=str(cer['certification']),
        required_skills=str(req['required_skills'])
    )
    for content, pos, loc, work, exp, emp, pay, deg, cer, req in zip(
        dev_contents, *([[json.loads(x) for x in dev_results]] * 9)
    )
]

test_examples_list = [
    dspy.Example(
        job_posting=content,
        position_title=str(pos['position_title']),
        location=str(loc['location']),
        work_arrangement=str(work['work_arrangement']),
        experience=str(exp['experience']),
        employment_type=str(emp['employment_type']),
        pay=str(pay['pay']),
        degree=str(deg['degree']),
        certifications=str(cer['certification']),
        required_skills=str(req['required_skills'])
    )
    for content, pos, loc, work, exp, emp, pay, deg, cer, req in zip(
        test_contents, *([[json.loads(x) for x in test_results]] * 9)
    )
]

In [14]:
trainset=train_examples_list
devset=dev_examples_list
testset = test_examples_list

trainset = [x.with_inputs('job_posting') for x in trainset]
devset = [x.with_inputs('job_posting') for x in devset]
testset = [x.with_inputs('job_posting') for x in testset]

In [15]:
print(train_examples['body'][0])

Derrickhand, Buckhannon, WV


Job Order Number
        
WV2925359


Post Date
        
05/12/2023


Job Location
        
Buckhannon, West Virginia 26201


County
        
Upshur


Job Summary
        
Job Summary: This position is a crew member assigned to work on a well service rig, responsible for performing 
services on oil and gas wells. Duties include performing all well-servicing tasks from an elevated position (rod 
basket or tubing board), assisting in rigging up or down, picking up or laying down tubing, and other functions 
specified by the customer or well operator. This position has a dotted line reporting line to: Rig Supervisor. 
Responsibilities: Assists the operator in rigging up and down, lining up the well service rig with the well. Sets 
hydraulic jacks, handles pads/boards and assists in attaching the guy wires to the anchor. Responsible for all 
elevated work associated with rigging up/down (i.e. removing horse head from pumping unit). Responsible for all 
work performed for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the 
elevator), performs servicing on the well. Drives the crew truck as needed. Operates tubing elevators for standing 
tubing in derrick. Assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the 
work floor or vice versa. Assists in walking the rods when laying down rods. Reports any safety hazards, accidents 
or maintenance issues to the rig supervisor. Ensures that work carried out is in compliance with company policies 
and procedures and according to safety regulations. May be required to work floors or operate the rig when needed. 
Performs other related duties as assigned. Preferred Qualifications: 1-2 years of Workover - Derrickhand experience
required. Ability to effectively communicate, both verbally and written. Ability to interact with others in a team 
environment. Ability to work in a fast-paced environment and handle multiple tasks at once. Basic problem solving 
and organizational skills. Excellent customer service skills, to provide world class value to customers CDL B 
license is required to drive rig. Must meet all qualifications defined in the Motor Vehicle Policy if required to 
drive. Ability to communicate verbally and in writing, in English, is preferred. Education Requirements: High 
school diploma, GED, or the equivalent is preferred. We are proud to offer a very competitive compensation and 
benefits package including: Medical Insurance Vision and Dental Insurance Life Insurance 401(k) Education 
assistance Short-Term Disability Paid time-off Request Priority Protected Veteran Referrals Equal Opportunity 
Employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender identity


Experience
        
0 Months


Pay Rate
        
0 $ / Hour


Master Group
        
Construction and Extraction Occupations


Job Type
        
Rotary Drill Operators, Oil and Gas


Shift
        
Day Shift

#### Test Uncompiled Module on combined Rouge Score

In [16]:
answ = trainset[0]
print(answ)

Example({'job_posting': 'Derrickhand, Buckhannon, WV\n\n\nJob Order Number\n        \nWV2925359\n\n\nPost Date\n   
\n05/12/2023\n\n\nJob Location\n        \nBuckhannon, West Virginia 26201\n\n\nCounty\n        \nUpshur\n\n\nJob 
Summary\n        \nJob Summary: This position is a crew member assigned to work on a well service rig, responsible 
for performing services on oil and gas wells. Duties include performing all well-servicing tasks from an elevated 
position (rod basket or tubing board), assisting in rigging up or down, picking up or laying down tubing, and other
functions specified by the customer or well operator. This position has a dotted line reporting line to: Rig 
Supervisor. Responsibilities: Assists the operator in rigging up and down, lining up the well service rig with the 
well. Sets hydraulic jacks, handles pads/boards and assists in attaching the guy wires to the anchor. Responsible 
for all elevated work associated with rigging up/down (i.e. removing horse head from pumping unit). Responsible for
all work performed for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the
elevator), performs servicing on the well. Drives the crew truck as needed. Operates tubing elevators for standing 
tubing in derrick. Assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the 
work floor or vice versa. Assists in walking the rods when laying down rods. Reports any safety hazards, accidents 
or maintenance issues to the rig supervisor. Ensures that work carried out is in compliance with company policies 
and procedures and according to safety regulations. May be required to work floors or operate the rig when needed. 
Performs other related duties as assigned. Preferred Qualifications: 1-2 years of Workover - Derrickhand experience
required. Ability to effectively communicate, both verbally and written. Ability to interact with others in a team 
environment. Ability to work in a fast-paced environment and handle multiple tasks at once. Basic problem solving 
and organizational skills. Excellent customer service skills, to provide world class value to customers CDL B 
license is required to drive rig. Must meet all qualifications defined in the Motor Vehicle Policy if required to 
drive. Ability to communicate verbally and in writing, in English, is preferred. Education Requirements: High 
school diploma, GED, or the equivalent is preferred. We are proud to offer a very competitive compensation and 
benefits package including: Medical Insurance Vision and Dental Insurance Life Insurance 401(k) Education 
assistance Short-Term Disability Paid time-off Request Priority Protected Veteran Referrals Equal Opportunity 
Employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender 
identity\n\n\nExperience\n        \n0 Months\n\n\nPay Rate\n        \n0 $ / Hour\n\n\nMaster Group\n        
\nConstruction and Extraction Occupations\n\n\nJob Type\n        \nRotary Drill Operators, Oil and Gas\n\n\nShift\n
\nDay Shift', 'position_title': 'Derrickhand', 'location': 'Buckhannon, West Virginia 26201', 'work_arrangement': 
'On-Site, Shifts', 'experience': '1-2 years of Derrickhand experience', 'employment_type': 'Full-time', 'pay': 'Not
specified', 'degree': 'High school diploma/GED or equivalent', 'certifications': 'CDL B License', 
'required_skills': 'Effective verbal/written communication in English, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}) (input_keys={'job_posting'})

In [17]:
with dspy.context(lm=llm):
    pred = uncompiled_module(trainset[0].job_posting)
    print(pred.info_extracted)

{
    'position_title': 'Derrickhand',
    'location': 'Buckhannon, West Virginia, WV 26201',
    'work_arrangement': 'On-site',
    'experience': '1-2 years',
    'employment_type': 'Full-time',
    'pay': 'Not specified.',
    'degree': 'High School',
    'certifications': 'CDL B License, 1-2 years Workover Derrickhand Experience, Basic Problem Solving Skills, 
Organizational Skills, Excellent Customer Service Skills, Motor Vehicle Policy Qualifications, English 
Communication',
    'required_skills': 'Derrickhand experience, communication, team interaction, fast pace, problem-solving, 
customer service, CDL B, basic math, high school diploma/GED.'
}

In [20]:
validate_ans(answ, pred)

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia 26201', 'work_arrangement': 'on-site, 
shifts', 'experience': '1-2 years of derrickhand experience', 'employment_type': 'full-time', 'pay': 'not 
specified', 'degree': 'high school diploma/ged or equivalent', 'certifications': 'cdl b license', 
'required_skills': 'effective verbal/written communication in english, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia, wv 26201', 'work_arrangement': 'on-site',
'experience': '1-2 years', 'employment_type': 'full-time', 'pay': 'not specified.', 'degree': 'high school', 
'certifications': 'cdl b license, 1-2 years workover derrickhand experience, basic problem solving skills, 
organizational skills, excellent customer service skills, motor vehicle policy qualifications, english 
communication', 'required_skills': 'derrickhand experience, communication, team interaction, fast pace, 
problem-solving, customer service, cdl b, basic math, high school diploma/ged.'}

0.5533506108848575

0.5533506108848575

#### BootStrap Few Shot With A Few Examples to Change the Output
    * Note for some reason compiling does an even worse job

In [21]:
teleprompter = BootstrapFewShot(metric=validate_ans) 
compiled = teleprompter.compile(uncompiled_module, trainset=trainset)
# teleprompter = BootstrapFewShotWithRandomSearch(metric=validate_ans, max_labeled_demos=16, max_rounds=1,  max_errors = 5, stop_at_score=0.50) 
# compiled = teleprompter.compile(uncompiled_module, trainset=trainset, valset = devset)

  0%|          | 0/20 [00:00<?, ?it/s]

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia 26201', 'work_arrangement': 'on-site, 
shifts', 'experience': '1-2 years of derrickhand experience', 'employment_type': 'full-time', 'pay': 'not 
specified', 'degree': 'high school diploma/ged or equivalent', 'certifications': 'cdl b license', 
'required_skills': 'effective verbal/written communication in english, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia, wv 26201', 'work_arrangement': 'on-site',
'experience': '1-2 years', 'employment_type': 'full-time', 'pay': 'not specified.', 'degree': 'high school', 
'certifications': 'cdl b license, 1-2 years workover derrickhand experience, basic problem solving skills, 
organizational skills, excellent customer service skills, motor vehicle policy qualifications, english 
communication', 'required_skills': 'derrickhand experience, communication, team interaction, fast pace, 
problem-solving, customer service, cdl b, basic math, high school diploma/ged.'}

0.5533506108848575

  5%|▌         | 1/20 [00:36<11:26, 36.15s/it]

...

 15%|█▌        | 3/20 [01:39<09:16, 32.73s/it]

{'position_title': 'graphic designer', 'location': 'goochland, va', 'work_arrangement': 'hybrid, with two in-office
days per week', 'experience': 'minimum 5 years design and publications experience', 'employment_type': 'part-time',
'pay': 'not specified', 'degree': 'college degree in graphic design, visual arts, or related field', 
'certifications': 'not specified', 'required_skills': 'proficiency with indesign, photoshop, illustrator, working 
knowledge of constant contact, strong organizational skills, excellent oral/written communication and 
client-relations skills, ability to work under pressure, working knowledge of ap style, 35mm and digital 
photography skills, mac environment'}

{'position_title': 'graphic designer (part-time)', 'location': 'goochland, va', 'work_arrangement': 'on-site', 
'experience': '5+ years', 'employment_type': 'part-time', 'pay': 'not specified', 'degree': "bachelor's in graphic 
design", 'certifications': 'indesign, photoshop, illustrator, constant contact, ap style, mac environment, direct 
mail, postal standards, photography skills', 'required_skills': 'graphic design, photoshop, illustrator, indesign, 
photography, branding, project management, communication, teamwork, time management, attention to detail, 
problem-solving, learning agility.'}

0.47476255088195385

 20%|██        | 4/20 [02:11<08:46, 32.89s/it]


#### Test Compiled Version

    * Issue with Compiler adds more verbose detail for some reason making it worst than out of the box

In [23]:
with dspy.context(lm=llm):
    pred = compiled(job_posting=trainset[0].job_posting)
    print(pred.info_extracted)

In [82]:
validate_ans(answ, pred)

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia 26201', 'work_arrangement': 'on-site, 
shifts', 'experience': '1-2 years of derrickhand experience', 'employment_type': 'full-time', 'pay': 'not 
specified', 'degree': 'high school diploma/ged or equivalent', 'certifications': 'cdl b license', 
'required_skills': 'effective verbal/written communication in english, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}

{'position_title': 'derrickhand\n\nthe derrickhand is a crew member assigned to work on a well service rig, 
responsible for performing services on oil and gas wells. this role involves various tasks such as assisting in 
rigging up and down, handling tubing, operating tubing elevators, walking rods, and other duties specified by the 
customer or well operator.\n\nkey responsibilities:\n\n1. **rigging up/down**: assists in setting hydraulic jacks, 
handling pads/boards, attaching guy wires to anchors, and performing tasks related to rigging up/down.\n2. 
**elevated work**: responsible for work associated with rigging up/down like removing the horse head from the 
pumping unit.\n3. **rod/tubing management**: performs tasks involving transferring rods and tubing between vertical
racks and elevators, servicing wells, driving crew trucks as needed, operating tubing elevators, picking up or 
laying down tubing, manually lifting tubing onto work floors, walking rods when laying them down, etc.\n\npreferred
qualifications:\n\n- 1-2 years of experience in a similar role (workover - derrickhand).\n- ability to communicate 
effectively both verbally and in writing.\n- team interaction skills.\n- fast-paced environment handling multiple 
tasks.\n- basic problem-solving and organizational skills.\n- excellent customer service skills for providing 
world-class value to customers.\n\nrequired qualifications:\n\n- cdl b license is required to drive the rig, with 
all qualifications defined by the motor vehicle policy met if driving is involved.\n- ability to communicate in 
english both verbally and written.\n\neducation requirements:\n\n- high school diploma, ged, or equivalent 
preferred.\n\ncompensation & benefits:\n\n- the derrickhand position offers a competitive compensation package 
including medical insurance, vision and dental insurance, life insurance, 401(k) plan, education assistance, 
short-term disability benefits, paid time off requests, priority for protected veterans referrals, and equal 
opportunity employment practices.\n\njob type: \n\n- rotary drill operators in the oil and gas industry.\n- shift 
type is day shift.', 'location': 'buckhannon, wv indicates that the job posting is for a location in west virginia,
usa. specifically, it mentions buckhannon which is a city in upshur county, west virginia. this information helps 
to pinpoint the geographical area where the job opportunity exists within the united states.', 'work_arrangement': 
'based on the information provided in the job posting for a derrickhand position:\n\n1. **job type**: the role 
involves working as a rotary drill operator in oil and gas operations.\n\n2. **shift schedule**: the work schedule 
specified is day shift, implying that the primary hours of operation are during daylight hours.\n\n3. **work 
location**: the job location is buckhannon, west virginia (wv), which indicates where the physical workplace or 
drilling site will be based.\n\n4. **job order number**: a specific job order number "wv2925359" is provided for 
tracking purposes within the organization\'s system.\n\n5. **qualifications and requirements**:\n   - a cdl b 
license is required to drive the rig, suggesting that transportation of equipment or materials might involve 
driving duties.\n   - the position requires meeting all qualifications defined in the motor vehicle policy if 
required to drive, emphasizing safety standards for drivers.\n   - effective communication skills are preferred, 
both verbally and in writing.\n\n6. **experience**: 1-2 years of workover-derrickhand experience is necessary, 
indicating that candidates should have prior experience in oilfield operations specifically related to derrickhands
or rig workers.\n\n7. **education requirements**: a high school diploma, ged, or equivalent is preferred as the 
minimum educational requirement for this position.\n\n8. **compensation and benefits**:\n   - the job offers a 
competitive compensation package including medic

0.027286214692330734

0.027286214692330734

In [83]:
compiled.save("compiled_v2_pydantic_v2.json")

#### Do Side by Side Comparieson on Evaluation versus uncompiled vs compiled
    * Note Library not fully developed something causing it to collect more information shows on results for uncompiled version

In [25]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10,return_outputs=True)

prev_score=evaluation(uncompiled_module, metric=validate_ans)

#improved_score=evaluation(compiled, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

{'position_title': 'senior inside sales rep/sales engineer', 'location': 'walpole, ma', 'work_arrangement': 
'hybrid', 'experience': 'depends on experience', 'employment_type': 'full time', 'pay': '$120k/year', 'degree': 'a 
bs in the engineering field', 'certifications': 'crm (salesforce.com), rfq experience and price quotes to the dod',
'required_skills': 'inside/outside technical sales experience, experience working with outside sales reps'}

{'position_title': 'federal sales engineer', 'location': 'walpole, ma', 'work_arrangement': 'hybrid remote', 
'experience': '5+ years', 'employment_type': 'full-time', 'pay': '$120k/year', 'degree': "bachelor's in 
engineering", 'certifications': 'crmf, salesforce.com, erp (epicor), power electronics certification', 
'required_skills': 'senior inside sales rep/sales engineer, power electronics/similar hardware experience, 
technical support dod/hs/prime contractors/commercial, dod selling experience (army preferred), 10-20% travel, 
opportunity identification/capture, hybrid schedule.'}

0.4114876385336743

Average Metric: 0.4114876385336743 / 1  (41.1):  10%|█         | 1/10 [00:20<03:06, 20.75s/it]

...

{'position_title': 'cryptocurrency / fx trader - entry level', 'location': 'not specified', 'work_arrangement': 
'remote', 'experience': 'no prior experience required', 'employment_type': 'full-time or part-time', 'pay': 
'results-based commissions and performance bonuses', 'degree': "bachelor's degree in finance, economics, or related
field preferred", 'certifications': 'none specified', 'required_skills': 'strong analytical skills, quick 
decision-making'}

{'position_title': 'cryptocurrency/fx trader - entry level', 'location': 'sugar land, texas, usa', 
'work_arrangement': 'remote', 'experience': 'no specified years of experience.', 'employment_type': 'contract', 
'pay': 'over $100k annually', 'degree': "bachelor's degree", 'certifications': 'no specified certifications.', 
'required_skills': 'entrepreneurial mindset, strong motivation, fast-paced adaptability, financial markets 
knowledge, risk management, quick decisions, analytical skills.'}

0.4759608665269042

Average Metric: 5.2328891428126445 / 10  (52.3): 100%|██████████| 10/10 [03:41<00:00, 22.10s/it]


,example_job_posting,position_title,location,work_arrangement,experience,employment_type,pay,degree,certifications,required_skills,pred_job_posting,info_extracted,validate_ans
0,"Federal Sales Engineer - Tech & ISR Experience - Hybrid Remote - Walpole MA Advanced Recruiting Solutions Walpole, MA Depends on Experience Full Time Work...",Senior Inside Sales Rep/Sales Engineer,"Walpole, MA",Hybrid,Depends on Experience,Full Time,$120K/year,A BS in the Engineering field,"CRM (salesforce.com), RFQ experience and price quotes to the DOD","Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps","federal sales engineer - tech & isr experience - hybrid remote - walpole ma advanced recruiting solutions walpole, ma depends on experience full time work...","{'position_title': 'Federal Sales Engineer', 'location': 'Walpole, MA', 'work_arrangement': 'Hybrid Remote', 'experience': '5+ years', 'employment_type': 'Full-time', 'pay': '$120k/year', 'degree': ""Bachelor's in Engineering"", 'certifications': 'CRMF, Salesforce.com, ERP...",✔️ [0.4114876385336743]
1,"Position Description Penetration Tester Location Washington, DC Req # 12763 # of openings 2 ECS is seeking a Penetration Tester to work in our Washington,...",Penetration Tester,"Washington, DC",On-site,10+ years of Penetration Testing experience,Full time,Not specified,Bachelors Degree in Computer Science,"Offensive Security certification (OSCP, OSCE), GIAC certification (GPEN, GWAPT, GXPN), or technology specific certification (MCSE, LPIC, CCNA)","NIST guidance, FedRAMP control baseline, industry best practice","position description penetration tester location washington, dc req # 12763 # of openings 2 ecs is seeking a penetration tester to work in our washington,...","{'position_title': 'Penetration Tester', 'location': 'Washington, DC', 'work_arrangement': 'On-site', 'experience': '10+', 'employment_type': 'Full-time', 'pay': 'Not specified', 'degree': ""Bachelor's"", 'certifications': 'OSCP, OSCE, GCIH, GWAPT, GCIA, GSEC', 'required_skills':...",✔️ [0.5930612244897959]
2,NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility Inc. compensation: HOURLY...,NURSES - RNs or LPNs,"Sudbury, MA 01776",on-site,Minimum of 1 year Long term care experience/SNF experience preferred,full-time,HOURLY - EVERY OTHER WEEKEND REQUIRED,Must have a valid MA Nursing License,RN or LPN License in Massachusetts,"Medication pass, treatments, resident care",nurses - rns & lpns in snf - sign on bonus - child daycare on site (sudbury) sudbury pines extended care facility inc. compensation: hourly...,"{'position_title': 'Nurse RN/LPN', 'location': 'Sudbury, MA', 'work_arrangement': 'On-site', 'experience': '1+ year', 'employment_type': 'Full-time, part-time', 'pay': 'Hourly', 'degree': 'Nursing License', 'certifications': 'RN license in Massachusetts, LPN...",✔️ [0.5849937343358396]
3,Planner IV - Transportation Planner Job Details Apply Print Share This listing closes on 7/17/2023 at 11:59 PM Pacific Time (US & Canada); Tijuana. Salary...,Planner IV - Transportation Planner,"Yakima, WA, 98901",On-site,5 years of increasingly responsible professional experience,Full-Time,$39.84 - $50.53 Hourly,Bachelor's Degree in Planning or other related field,None specified,"Transportation planning, coordination with the Yakama Nation, preparation of loans and grants",planner iv - transportation planner job details apply print share this listing closes on 7/17/2023 at 11:59 pm pacific time (us & canada); tijuana. salary...,"{'position_title': 'Transportation Planner IV', 'location': 'Yakima, WA', 'work_arrangement': 'On-site', 'experience': '5+ years', 'employment_type': 'Full-time', 'pay': '$39.84 - $50.53 hourly', 'degree': ""Bachelor's degree"", 'certifications': 'Transportation Planning,...",✔️ [0.5855932203389831]
4,"Associate Attorney Juan Ramos Law Group, PLLC McAllen, TX Job Details Full-time From $50,000 a year 1 day ago Qualif